# Tier 0 — every paper figure from the tracked results, no data needed

This notebook regenerates the three main figures and the main SI tables of the
paper from the CSV/JSON summaries that are tracked in this repository. It needs
only `pandas`, `numpy` and `matplotlib` (no `cropchoice` install, no cluster, no
LSMS data). Runtime: about one minute.

Every cell states which tracked file it reads, so a number in the paper can be
traced back to a file in `git`.

Country colours are fixed by median plot income (Malawi, Ethiopia, Uganda,
Tanzania, Nigeria, Mali) and are the same in every figure.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# repository root = the first parent that contains 08_BIRL_v2/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "08_BIRL_v2").is_dir())
OUT08 = ROOT / "08_BIRL_v2" / "outputs"
RES07 = ROOT / "07_2050_Counter_Fact" / "results"
FIG_DIR = ROOT / "notebooks" / "figures_tier0"
FIG_DIR.mkdir(exist_ok=True)

COUNTRIES = ["Malawi", "Ethiopia", "Uganda", "Tanzania", "Nigeria", "Mali"]   # by median plot income
ISO = {"Malawi": "MW", "Ethiopia": "ET", "Uganda": "UG", "Tanzania": "TZ", "Nigeria": "NG", "Mali": "ML"}
COLOR = dict(zip(COUNTRIES, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))  # validated CVD-safe order
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
                     "font.size": 7, "axes.titlesize": 8, "axes.labelsize": 7, "xtick.labelsize": 6.5, "ytick.labelsize": 6.5,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.6,
                     "xtick.major.width": 0.5, "ytick.major.width": 0.5, "legend.frameon": False, "figure.dpi": 130})
MM = 1 / 25.4
print("repository:", ROOT)

## Inputs

| object | tracked file |
|---|---|
| country-level `a, b, c` and `sigma*` | `08_BIRL_v2/outputs/semipar/summary.csv`, `derived.csv` |
| asset-tercile `a, b, c` | `08_BIRL_v2/outputs/semipar_assets/summary.csv`, `derived.csv` |
| quantile-weight test | `08_BIRL_v2/outputs/exp_risk_signal_tmp/exp_risk_signal/exp_risk_signal.json` |
| utility-family sweep | `08_BIRL_v2/outputs/exp_utility*_tmp/*/*.json` |
| CRRA variant (rho at its bounds) | `08_BIRL_v2/outputs/v2_country/summary.csv` |
| climate and policy counterfactual | `07_2050_Counter_Fact/results/choice_cf/tables/*.csv` |
| Sobol sweep | `07_2050_Counter_Fact/results/choice_cf_sobol/summary.json`, `sobol_indices.csv` |
| simulation recovery, leakage, familiarity | `08_BIRL_v2/outputs/recovery/nuts/report.json`, `exp_leak_tmp/…`, `exp_familiar_tmp/…` |

In [ ]:
semipar = pd.read_csv(OUT08 / "semipar" / "summary.csv")
derived = pd.read_csv(OUT08 / "semipar" / "derived.csv").set_index("country")
assets = pd.read_csv(OUT08 / "semipar_assets" / "summary.csv")
assets_d = pd.read_csv(OUT08 / "semipar_assets" / "derived.csv").set_index("country")
risk = json.load(open(OUT08 / "exp_risk_signal_tmp" / "exp_risk_signal" / "exp_risk_signal.json"))
util = {n: json.load(open(OUT08 / f"exp_utility{s}_tmp" / f"exp_utility{s}" / f"exp_utility{s}.json"))["results"]
        for n, s in [("level", ""), ("log", "_log"), ("sf", "_sf")]}
crra = pd.read_csv(OUT08 / "v2_country" / "summary.csv")
tables = {p.stem: pd.read_csv(p) for p in (RES07 / "choice_cf" / "tables").glob("*.csv")}
sobol = json.load(open(RES07 / "choice_cf_sobol" / "summary.json"))
sobol_idx = pd.read_csv(RES07 / "choice_cf_sobol" / "sobol_indices.csv")
M_C = {c: float(v) for c, v in sobol["m_c"].items()}
print("loaded:", ", ".join(sorted(tables)), "| countries:", COUNTRIES)


def country_rows(df, param, col="median"):
    d = df[df.param == param].set_index("country").reindex(COUNTRIES)
    return d[col].to_numpy(), d["hpdi_lo"].to_numpy(), d["hpdi_hi"].to_numpy()

## Figure 1 — what the choice data identify

**a** The estimated value of a crop's predicted income dispersion,
$b\,\sigma + c\,\sigma^2$, per country (posterior medians): a U shape with its
minimum $\sigma^*$ between 1.1 and 1.3 everywhere. **b** The country-level
coefficients $a$ (level), $b$, $c$ with 89% HPDIs (diamonds) and the same
coefficients estimated separately for the poorest, middle and richest asset
tercile within each country (dots, left to right). **c** Weights on the three
predicted income quantiles in a choice model with crop fixed effects: choices load
on the 10th percentile. **d** Log-likelihood gain per observation of each
parametric utility family relative to the risk-neutral reference, against the free
benchmarks. **e** The CRRA alternative: $\rho$ sits at its prior bounds.

In [ ]:
fig = plt.figure(figsize=(183 * MM, 150 * MM))
gs = fig.add_gridspec(3, 6, height_ratios=[1.25, 1.0, 1.0], hspace=0.65, wspace=0.55)

# a: U-shaped dispersion value
ax = fig.add_subplot(gs[0, :3])
sig = np.linspace(0.05, 2.5, 200)
b_med, _, _ = country_rows(semipar, "b"); c_med, _, _ = country_rows(semipar, "c")
for cn, b, c in zip(COUNTRIES, b_med, c_med):
    v = b * sig + c * sig ** 2
    ax.plot(sig, v, color=COLOR[cn], lw=1.0)
    s_star = derived.loc[cn, "sigma_star"]
    ax.plot(s_star, b * s_star + c * s_star ** 2, "o", ms=3.5, color=COLOR[cn])
ax.axhline(0, color="#999999", lw=0.5)
ax.set_xlim(0, 2.6); ax.set_xlabel("predicted dispersion of log plot income, $\\sigma$")
ax.legend([matplotlib.lines.Line2D([], [], color=COLOR[c], lw=1.2) for c in COUNTRIES], COUNTRIES, loc="upper left", fontsize=5.5, ncol=2, handlelength=1.2)
ax.set_ylabel("$b\\,\\sigma + c\\,\\sigma^2$ (logit units)")
ax.set_title("a  Dispersion is penalised up to $\\sigma^*\\approx$ 1.1–1.3 (dots)", loc="left", fontweight="bold")

# b: a, b, c with asset terciles
for j, (param, ttl) in enumerate([("a", "$a$: level $\\mu$"), ("b", "$b$: dispersion $\\sigma$"), ("c", "$c$: $\\sigma^2$")]):
    ax = fig.add_subplot(gs[0, 3 + j])
    med, lo, hi = country_rows(semipar, param)
    x = np.arange(len(COUNTRIES))
    ax.axhline(0, color="#999999", lw=0.5)
    for i, cn in enumerate(COUNTRIES):
        ax.errorbar(x[i], med[i], yerr=[[med[i] - lo[i]], [hi[i] - med[i]]], fmt="D", ms=4, color=COLOR[cn], lw=0.7, zorder=3)
        t = assets[(assets.param == param) & (assets.country == cn)].sort_values("tercile")
        for k, (_, r) in enumerate(t.iterrows()):
            ax.errorbar(x[i] - 0.28 + 0.28 * k, r["median"], yerr=[[r["median"] - r["hpdi_lo"]], [r["hpdi_hi"] - r["median"]]],
                        fmt="o", ms=1.8, color=COLOR[cn], alpha=0.75, lw=0.5)
    ax.set_xticks(x); ax.set_xticklabels([ISO[c] if i % 2 == 0 else "\n" + ISO[c] for i, c in enumerate(COUNTRIES)])
    ax.set_title(("b  " if j == 0 else "") + ttl, loc="left", fontweight="bold" if j == 0 else None)
    if j == 0:
        ax.set_ylabel("posterior median, 89% HPDI")
fig.text(0.75, 0.585, "diamond: all households; dots: asset terciles, poorest to richest (left to right)", ha="center", fontsize=5.5, color="#555555")

# c: quantile weights
pc = risk["B_q3"]["per_country"]
for i, cn in enumerate(COUNTRIES):
    ax = fig.add_subplot(gs[1, i])
    w = [pc[cn]["w10"], pc[cn]["w50"], pc[cn]["w90"]]; se = pc[cn]["se"]
    ax.bar([0, 1, 2], w, yerr=se, color=COLOR[cn], width=0.6, error_kw={"lw": 0.5})
    ax.axhline(0, color="#999999", lw=0.5)
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["$q_{10}$", "$q_{50}$", "$q_{90}$"])
    ax.set_title(("c  " if i == 0 else "") + cn, loc="left", fontweight="bold" if i == 0 else None, fontsize=7)
    if i == 0:
        ax.set_ylabel("weight per unit\nof median income")
fig.text(0.5, 0.345, "weights on three predicted income quantiles (choice model with crop fixed effects, ±1 s.e.): choices load on the 10th percentile",
         ha="center", fontsize=5.5, color="#555555")

# d: utility families (log-lik gain per obs vs risk-neutral reference of the same space)
N_OBS = 222023
fam = [("Free (μ, σ, σ²) (benchmark)", util["log"]["logfree"]["loglik"] - util["log"]["logev"]["loglik"]),
       ("Free node weights (benchmark)", util["level"]["free5"]["loglik"] - util["level"]["ev"]["loglik"]),
       ("Rank-dependent, Prelec weighting", util["level"]["rdeu_prelec"]["loglik"] - util["level"]["ev"]["loglik"]),
       ("Expected shortfall", util["level"]["es"]["loglik"] - util["level"]["ev"]["loglik"]),
       ("CRRA, lognormal closed form", util["log"]["logcrra"]["loglik"] - util["log"]["logev"]["loglik"]),
       ("Rank-dependent, power weighting", util["level"]["rdeu_pow"]["loglik"] - util["level"]["ev"]["loglik"]),
       ("Ruin-probability penalty", util["sf"]["ruin"]["loglik"] - util["sf"]["logev"]["loglik"]),
       ("Mean minus θ·sd (log)", util["log"]["logmsd"]["loglik"] - util["log"]["logev"]["loglik"]),
       ("Roy safety-first ratio", util["sf"]["roy"]["loglik"] - util["sf"]["logev"]["loglik"])]
ax = fig.add_subplot(gs[2, :4])
y = np.arange(len(fam))[::-1]
for yi, (name, g) in zip(y, fam):
    col = "#111111" if "benchmark" in name else "#9a9a9a"
    ax.plot([0, g / N_OBS], [yi, yi], color=col, lw=1.2); ax.plot(g / N_OBS, yi, "o", ms=3, color=col)
    if "benchmark" in name:
        ax.text(g / N_OBS + 0.0008, yi, f"{g / N_OBS:.3f}", va="center", fontsize=6)
ax.set_yticks(y); ax.set_yticklabels([n for n, _ in fam]); ax.axvline(0, color="#999999", lw=0.5)
ax.set_xlabel("log-likelihood gain per observation vs risk-neutral reference")
ax.set_title("d  No parametric family reaches the free benchmark", loc="left", fontweight="bold")

# e: CRRA rho at its bounds
ax = fig.add_subplot(gs[2, 4:])
rho = crra[crra.param == "rho_c"].set_index("country").reindex(COUNTRIES)
for i, cn in enumerate(COUNTRIES):
    ax.plot([i, i], [rho.loc[cn, "hpdi_lo"], rho.loc[cn, "hpdi_hi"]], color=COLOR[cn], lw=1.2)
    ax.plot(i, rho.loc[cn, "median"], "o", ms=3.5, color=COLOR[cn])
ax.axhline(5.0, color="#999999", lw=0.5); ax.axhline(0.1, color="#999999", lw=0.5)
ax.text(len(COUNTRIES) - 0.6, 4.55, "prior bounds", fontsize=5.5, color="#777777", ha="right")
ax.set_xticks(range(len(COUNTRIES))); ax.set_xticklabels([ISO[c] for c in COUNTRIES]); ax.set_ylabel("CRRA $\\rho$")
ax.set_title("e  The CRRA alternative: $\\rho$ pinned at its bounds", loc="left", fontweight="bold")
fig.savefig(FIG_DIR / "fig1_identification.pdf", bbox_inches="tight")
plt.show()

## Figure 2 — 2050 climate: outcomes and crop shifts

**a** Change in expected plot income, in the probability that income falls below
30% of the country median, and in the expected shortfall, under SSP2-4.5 and
SSP5-8.5 (posterior medians; blue = improvement, red = deterioration).
**b** Crop shares today and under SSP5-8.5 without policy; crops that move by at
least one percentage point are named.

In [ ]:
ms = tables["metrics_summary"]
def med(metric, climate, policy="none"):
    d = ms[(ms.metric == metric) & (ms.climate == climate) & (ms.policy == policy)].set_index("country")
    return d["median"].reindex(COUNTRIES)

cols = []
for clim in ["ssp245", "ssp585"]:
    inc = 100 * (med("exp_income", clim) / med("exp_income", "baseline") - 1)
    pb = 100 * (med("p_below_theta", clim) - med("p_below_theta", "baseline"))
    sf = 100 * (med("exp_shortfall", clim) / med("exp_shortfall", "baseline") - 1)
    cols += [(clim, "income (%)", inc, +1), (clim, "P(<30%) (pp)", pb, -1), (clim, "shortfall (%)", sf, -1)]

fig = plt.figure(figsize=(183 * MM, 120 * MM))
gs = fig.add_gridspec(2, 6, height_ratios=[1.0, 1.0], hspace=0.7, wspace=0.35)
ax = fig.add_subplot(gs[0, :3])
mat = np.column_stack([c[2].to_numpy() for c in cols]); good = np.array([c[3] for c in cols])
norm = matplotlib.colors.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)
scaled = np.column_stack([np.clip(mat[:, j] * good[j] / max(np.nanmax(np.abs(mat[:, j])), 1e-9), -1, 1) for j in range(mat.shape[1])])
ax.imshow(scaled, cmap="RdBu", norm=norm, aspect="auto")
for i in range(len(COUNTRIES)):
    for j in range(mat.shape[1]):
        ax.text(j, i, f"{mat[i, j]:+.2g}" if abs(mat[i, j]) < 10 else f"{mat[i, j]:+.0f}", ha="center", va="center", fontsize=6,
                color="white" if abs(scaled[i, j]) > 0.65 else "#111111")
ax.set_yticks(range(len(COUNTRIES))); ax.set_yticklabels(COUNTRIES)
ax.set_xticks(range(len(cols))); ax.set_xticklabels([c[1] for c in cols], fontsize=5.5)
ax.axvline(2.5, color="white", lw=2)
ax.text(1, -0.9, "SSP2-4.5, 2050", ha="center", fontsize=6.5, fontweight="bold"); ax.text(4, -0.9, "SSP5-8.5, 2050", ha="center", fontsize=6.5, fontweight="bold")
ax.set_title("a  Change vs current climate (blue = improvement, red = deterioration)", loc="left", fontweight="bold")
for s in ("top", "right", "left", "bottom"):
    ax.spines[s].set_visible(False)

# b: crop-share slopes
cs = tables["crop_shifts"]
d = cs[(cs.climate == "ssp585") & (cs.policy == "none")]
for i, cn in enumerate(COUNTRIES):
    ax = fig.add_subplot(gs[1, i])
    dd = d[d.country == cn]
    for _, r in dd.iterrows():
        y0 = 100 * r["base_share_median"]; y1 = y0 + 100 * r["delta_share_median"]
        big = abs(100 * r["delta_share_median"]) >= 1
        ax.plot([0, 1], [y0, y1], color=COLOR[cn] if big else "#c9c9c9", lw=1.2 if big else 0.7, marker="o", ms=1.8)
        if big:
            ax.text(1.05, y1, f"{r['crop'].replace('_', '/')} {100 * r['delta_share_median']:+.0f}", fontsize=5, va="center")
    ax.set_xlim(-0.2, 2.0); ax.set_xticks([0, 1]); ax.set_xticklabels(["today", "2050"])
    ax.set_title(("b  " if i == 0 else "") + cn, loc="left", fontweight="bold" if i == 0 else None, fontsize=7)
    if i == 0:
        ax.set_ylabel("share of plots (%)")
fig.text(0.5, 0.02, "crop shares today and under SSP5-8.5 (2050), no policy; crops that move by at least 1 pp are named", ha="center", fontsize=5.5, color="#555555")
fig.savefig(FIG_DIR / "fig2_climate.pdf", bbox_inches="tight")
plt.show()

## Figure 3 — policy: level versus dispersion, and the cost of protection

**a** Share of plots switching crop or intensity under a pure variance cut divided
by the share switching under an income transfer (SSP5-8.5). Dots and bars: posterior
median and 89% HPDI at the base policy sizes; boxes: interquartile range and 5–95%
range across the Sobol sweep of policy sizes. **b** Share of plots switching under
each instrument. **c** Expected-shortfall reduction per public dollar for the income
floor and for index insurance under two cost accountings. **d** Change in expected
income against change in expected shortfall for the four real instruments.

In [ ]:
hr = tables["headline_ratio"]; hr = hr[hr.climate == "ssp585"].set_index("country").reindex(COUNTRIES)
hu = sobol["headline_uncertainty"]
pe = tables["policy_effects"]; pe5 = pe[pe.climate == "ssp585"]
POL = ["transfer", "contraction", "safety_net", "insurance", "both"]
POL_LABEL = {"transfer": "income\ntransfer", "contraction": "variance\ncut", "safety_net": "income\nfloor", "insurance": "index\ninsurance", "both": "floor +\ninsurance"}
MARK = {"transfer": "o", "safety_net": "s", "insurance": "D", "both": "P"}

fig = plt.figure(figsize=(183 * MM, 135 * MM))
gs = fig.add_gridspec(2, 2, hspace=0.55, wspace=0.35)

ax = fig.add_subplot(gs[0, 0])
for i, cn in enumerate(COUNTRIES):
    q05, q95 = hu[cn]["param_box_q05_q95"]; iqr = hu[cn]["param_box_iqr"]; pm = hu[cn]["param_box_median"]
    ax.plot([i, i], [q05, q95], color=COLOR[cn], lw=0.6, alpha=0.6)
    ax.add_patch(matplotlib.patches.Rectangle((i - 0.12, pm - iqr / 2), 0.24, iqr, color=COLOR[cn], alpha=0.25, lw=0))
    ax.errorbar(i, hr.loc[cn, "ratio_median"], yerr=[[hr.loc[cn, "ratio_median"] - hr.loc[cn, "ratio_lo"]], [hr.loc[cn, "ratio_hi"] - hr.loc[cn, "ratio_median"]]],
                fmt="o", ms=3.5, color=COLOR[cn], lw=0.8)
    ax.text(i + 0.18, hr.loc[cn, "ratio_median"], f"{hr.loc[cn, 'ratio_median']:.1f}", fontsize=6, va="center")
ax.axhline(1, color="#999999", lw=0.5); ax.set_yscale("log"); ax.set_yticks([0.25, 0.5, 1, 2, 4]); ax.set_yticklabels(["0.25", "0.5", "1", "2", "4"])
ax.set_xticks(range(len(COUNTRIES))); ax.set_xticklabels([f"{c}\n{M_C[c]:.0f}" for c in COUNTRIES])
ax.set_xlabel("country (median plot income, USD)"); ax.set_ylabel("plots switching: variance cut / income transfer")
ax.set_title("a  Dispersion moves the poorest, income level moves the richest", loc="left", fontweight="bold")

ax = fig.add_subplot(gs[0, 1])
sw = ms[(ms.metric == "switch_share") & (ms.climate == "ssp585")]
for j, pol in enumerate(POL):
    for i, cn in enumerate(COUNTRIES):
        v = float(sw[(sw.policy == pol) & (sw.country == cn)]["median"].iloc[0])
        ax.scatter(j, i, s=900 * v, color=COLOR[cn], alpha=0.9); ax.text(j + 0.33, i, f"{100 * v:.0f}", va="center", fontsize=6)
ax.set_xticks(range(len(POL))); ax.set_xticklabels([POL_LABEL[p] for p in POL], fontsize=6)
ax.set_yticks(range(len(COUNTRIES))); ax.set_yticklabels(COUNTRIES); ax.invert_yaxis(); ax.set_xlim(-0.5, len(POL) - 0.3)
for s in ("top", "right", "left", "bottom"):
    ax.spines[s].set_visible(False)
ax.set_title("b  Plots switching crop or intensity (%), SSP5-8.5", loc="left", fontweight="bold")

ax = fig.add_subplot(gs[1, 0])
def es_delta(pol):
    return (med("exp_shortfall", "ssp585", pol) - med("exp_shortfall", "ssp585", "none"))
cost = {p: med("cost", "ssp585", p) for p in ["safety_net", "insurance"]}
prem = med("premium", "ssp585", "insurance")
floor_pd = -es_delta("safety_net") / cost["safety_net"] * 100
ins_load = -es_delta("insurance") / cost["insurance"] * 100
ins_full = -es_delta("insurance") / prem * 100
for i, cn in enumerate(COUNTRIES):
    ax.plot([floor_pd[cn], ins_full[cn]], [i, i], color=COLOR[cn], lw=0.8)
    ax.plot(floor_pd[cn], i, "s", ms=4, color=COLOR[cn]); ax.plot(ins_full[cn], i, "o", ms=4, color=COLOR[cn]); ax.plot(ins_load[cn], i, "o", ms=4, mfc="none", color=COLOR[cn])
ax.set_xscale("log"); ax.set_yticks(range(len(COUNTRIES))); ax.set_yticklabels(COUNTRIES); ax.invert_yaxis()
ax.set_xlabel("expected-shortfall reduction per public dollar (×100)")
ax.plot([], [], "s", color="#555555", label="income floor"); ax.plot([], [], "o", color="#555555", label="insurance, full premium as cost"); ax.plot([], [], "o", mfc="none", color="#555555", label="insurance, loading only as cost")
ax.legend(loc="lower right", fontsize=5.5)
ax.set_title("c  Floor vs insurance: the ranking is a cost-accounting choice", loc="left", fontweight="bold")

ax = fig.add_subplot(gs[1, 1])
for pol in ["transfer", "safety_net", "insurance", "both"]:
    dy = med("exp_income", "ssp585", pol) - med("exp_income", "ssp585", "none"); dx = es_delta(pol)
    for cn in COUNTRIES:
        ax.plot(dx[cn], dy[cn], MARK[pol], ms=4, color=COLOR[cn], mec="white", mew=0.4)
    ax.plot([], [], MARK[pol], color="#555555", label=POL_LABEL[pol].replace("\n", " "))
ax.axhline(0, color="#999999", lw=0.5); ax.axvline(0, color="#999999", lw=0.5)
ax.set_xlabel("change in expected shortfall (USD per plot-season)"); ax.set_ylabel("change in expected income (USD per plot-season)")
ax.legend(loc="lower left", fontsize=5.5, ncol=2); ax.text(0.98, 0.95, "colour = country", transform=ax.transAxes, ha="right", fontsize=5.5, color="#777777")
ax.set_title("d  Protection lowers the downside and, through crop choice, the mean", loc="left", fontweight="bold")
fig.savefig(FIG_DIR / "fig3_policy.pdf", bbox_inches="tight")
plt.show()

## SI tables

The same tracked files, shown as tables: country parameters, asset-tercile contrasts,
the CRRA variant, the utility-family sweep, Sobol total indices for the headline
ratio, the simulation-recovery verdict, and the leakage / familiarity checks.

In [ ]:
t1 = pd.DataFrame({p: [f"{m:.2f} [{lo:.2f}, {hi:.2f}]" for m, lo, hi in zip(*country_rows(semipar, p))] for p in ["a", "b", "c"]}, index=COUNTRIES)
t1["sigma*"] = [f"{derived.loc[c, 'sigma_star']:.2f} [{derived.loc[c, 'sigma_star_lo']:.2f}, {derived.loc[c, 'sigma_star_hi']:.2f}]" for c in COUNTRIES]
display(t1.rename_axis("Table 1: country-level coefficients (median [89% HPDI])"))

In [ ]:
t_assets = assets_d.reindex(COUNTRIES)[["da_poor_minus_rich", "da_lo", "da_hi", "p_da_gt0", "db_poor_minus_rich", "db_lo", "db_hi", "p_db_gt0"]].round(2)
display(t_assets.rename_axis("Asset terciles: poorest minus richest (a and b)"))

In [ ]:
rho = crra[crra.param.isin(["rho_c", "s_c", "beta_c"])].pivot(index="country", columns="param", values="median").reindex(COUNTRIES).round(3)
display(rho.rename_axis("CRRA / Stone-Geary variant: posterior medians (rho at its bounds 0.1 / 5.0)"))

In [ ]:
display(pd.DataFrame(fam, columns=["family", "log-lik gain vs risk-neutral"]).assign(**{"per observation": lambda d: d["log-lik gain vs risk-neutral"] / N_OBS}).round(4))

In [ ]:
st = sobol_idx[sobol_idx.metric == "headline_ratio"].pivot(index="param", columns="country", values="ST").reindex(columns=COUNTRIES).round(2)
display(st.rename_axis("Sobol total index of the headline ratio, by policy parameter"))
print("share of the parameter box with 'low-income four > 1 and Mali < 1':", round(sobol["share_points_low4_gt1_and_mali_lt1"], 3))

In [ ]:
rec = json.load(open(OUT08 / "recovery" / "nuts" / "report.json"))
print("simulation recovery (NUTS):", rec.get("verdict"), "| tolerances met:",
      {k: f"{v['tolerances_met']}/{v['tolerances_judged']}" for k, v in rec["sets"].items()})
leak = json.load(open(OUT08 / "exp_leak_tmp" / "exp_leak" / "exp_leak.json"))
fam_ = json.load(open(OUT08 / "exp_familiar_tmp" / "exp_familiar" / "exp_familiar.json"))
print("leakage test keys:", list(leak.keys())[:4], "| familiarity test keys:", list(fam_.keys())[:4])
print("figures written to", FIG_DIR)